# **Hybrid Music Recommender (PySpark)**
### Vrishabh Gupta · CIS 415 course project

Build and test with **small data in Google Colab** (or locally), then run this exact notebook on **GCP Dataproc** with the big data. The only line that should change between the two runs is `data_file_name` in the config cell.

**What this notebook does**

1. Loads the modelling table: real MLHD+ listening history joined to AcousticBrainz audio features.
2. Builds a proper cold-start test set. The dataset's negatives were all drawn from recordings that existed before the split, so a brand-new recording is always a positive. That makes a naive cold-start test rigged, so I pair each new recording a listener played with four other new recordings they didn't.
3. Removes two leaks. A training row's own plays are inside its item's popularity counts, and an ALS model scores the very rows it trained on. Both get fixed before any model sees them.
4. Builds the two engines: collaborative filtering (ALS) and content-based filtering (audio similarity to the listener's taste).
5. Trains 7 models: popularity baseline, CF alone, CBF alone, a score-level blend, and three feature-level hybrids (Logistic Regression, Random Forest, Gradient-Boosted Trees).
6. Evaluates on **warm** recordings (known before the split) and **cold** recordings (brand new after it) separately.

Run the cells top to bottom. On small data the whole thing takes 15 to 20 minutes in Colab, most of it training the Gradient-Boosted Trees.

In [ ]:
# Check if the code is running in Colab
if 'google.colab' in str(get_ipython()):
  print('Running on CoLab')
  RunningInColab = True
else:
  print('Not running on CoLab')
  RunningInColab = False

In [ ]:
# Configure the following to use YOUR GCP setup

# 1. Project ID (not the Project Name!!!)
project_id = 'your-gcp-project-id'

# 2. Cloud Storage bucket name
bucket = 'your-bucket-name'

# 3. Folder inside the bucket that holds the project data
path_to_data_files = "/data_for_project/"

# 4. Pick the dataset
#       Google Colab: build and test with SMALL data
#       GCP Dataproc: run SMALL first to confirm the setup works, then switch to BIG
#    This is the ONLY line that should change between the small run and the big run.

data_file_name = "small_dataset.parquet"
# data_file_name = "big_dataset.parquet"      # a folder of 16 part files, upload the whole folder

# 5. Running locally or without Cloud Storage? Point this at the repo's data folder, e.g. "../data/".
#    Leave it as None to read from the bucket above.
LOCAL_DATA_DIR = None

# Build the full path (no need to change this)
relative_path_to_file = path_to_data_files[1:] + data_file_name
full_file_path = "gs://" + bucket + "/" + relative_path_to_file
if LOCAL_DATA_DIR:
  full_file_path = LOCAL_DATA_DIR + data_file_name

print(f"ProjectID (and not the Project Name) is: {project_id}")
print(f"Bucket name is: {bucket}")
print(f"Relative path to the data is {relative_path_to_file}")
print(f"Full path to the data is {full_file_path}")

In [ ]:
# Check if PySpark is available and install it if not.
# In GCP Dataproc this should NOT run the installation.
try:
  from pyspark.sql import SparkSession
  pyspark_available = 'Y'
except:
  pyspark_available = 'N'

if RunningInColab == True or pyspark_available == "N":
  print("Installing PySpark...")
  !apt-get update -qq
  !apt-get install -y openjdk-11-jdk-headless -qq
  !pip install -q pyspark
  print("PySpark installed")
else:
  print("PySpark already pre-installed!")

In [ ]:
# Authorize access from Colab to GCP Cloud Storage (skipped when reading a local copy)
if RunningInColab == True and not LOCAL_DATA_DIR:
  from google.cloud import storage
  import google.auth

  !pip install gcsfs
  import gcsfs

  from google.colab import auth
  auth.authenticate_user()

  credentials, default_project_id = google.auth.default()
  !gcloud config set project {project_id}

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.recommendation import ALS
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
import pandas as pd
import time

if RunningInColab == True:
  spark = (SparkSession.builder.master("local[*]")
           .config("spark.driver.memory", "4g")
           .config("spark.sql.shuffle.partitions", "16")
           .getOrCreate())
  print("Spark Session created in Colab")
  from google.cloud import storage
elif LOCAL_DATA_DIR:
  spark = (SparkSession.builder.master("local[*]")
           .config("spark.driver.memory", "4g")
           .config("spark.sql.shuffle.partitions", "16")
           .getOrCreate())
  print("Local Spark session created")
else:
  # Dataproc already created a session called `spark`; getOrCreate just hands that one back
  spark = SparkSession.builder.getOrCreate()
  print("Using the Spark session provided by Dataproc")

SEED = 42          # fixed everywhere, so every run makes the same splits, folds and samples
print("Spark version:", spark.version)

In [ ]:
# Read the data
print(f"Reading: {full_file_path}")

if RunningInColab == True and not LOCAL_DATA_DIR:
  # Colab: copy from Cloud Storage to the Colab machine, then read the local copy.
  # Works for a single file (small_dataset.parquet) or a folder of part files (big_dataset.parquet).
  import os, shutil
  storage_client = storage.Client()
  local_path = "/content/" + data_file_name
  blobs = [b for b in storage_client.list_blobs(bucket, prefix=relative_path_to_file)
           if not b.name.endswith("/") and not os.path.basename(b.name).startswith(".")]

  if len(blobs) == 0:
    # Nothing at that path. Look for the file anywhere in the bucket and say where it actually is.
    everything = [b.name for b in storage_client.list_blobs(bucket)]
    folders = sorted({n.split(data_file_name)[0] for n in everything if data_file_name in n})
    print(f"\nNothing found at gs://{bucket}/{relative_path_to_file}")
    if folders:
      print(f"But {data_file_name} IS in the bucket. Set this in the config cell, re-run it, then re-run this cell:")
      for f in folders:
        print(f'    path_to_data_files = "/{f}"')
    else:
      print(f"{data_file_name} isn't anywhere in bucket '{bucket}', so it still needs uploading (see docs/running_on_gcp.md).")
      print("The bucket currently holds:")
      for n in everything[:40]:
        print("   ", n)
    raise FileNotFoundError(f"gs://{bucket}/{relative_path_to_file} not found")

  # clear out anything left over from an earlier attempt
  if os.path.isdir(local_path):
    shutil.rmtree(local_path)
  elif os.path.exists(local_path):
    os.remove(local_path)

  if len(blobs) == 1 and blobs[0].name == relative_path_to_file:
    blobs[0].download_to_filename(local_path)
  else:
    os.makedirs(local_path, exist_ok=True)
    for b in blobs:
      b.download_to_filename(os.path.join(local_path, os.path.basename(b.name)))
  print(f"Downloaded {len(blobs)} file(s) to {local_path}")
  read_path = local_path
else:
  # Dataproc reads straight from Cloud Storage; a local run reads LOCAL_DATA_DIR
  read_path = full_file_path

if data_file_name.endswith(".csv"):
  raw_df = spark.read.csv(read_path, header=True, inferSchema=True)
else:
  raw_df = spark.read.parquet(read_path)

raw_df.show(5)
print(f"Total number of records = {raw_df.count():,}")

In [ ]:
# What came in
raw_df.printSchema()
raw_df.groupBy("split", "played").count().orderBy("split", "played").show()

## 1. A fair cold-start test set

Every negative in the dataset is a recording that existed before the split date, so any recording that first appears *after* the split is always a positive. Scoring cold-start on that would be rigged: "never heard of it" would mean "played".

Fix: for each brand-new recording a listener played, pick four **other** brand-new recordings that listener did not play, and label those 0. Now popularity is zero on both sides of the cold test and only content and taste can separate them. The picks are hash-based, so every run (Colab or Dataproc) makes the same ones.

In [ ]:
K_COLD = 4

cold_pos = raw_df.filter((F.col("split") == "test") & (F.col("played") == 1)
                         & (F.col("recording_listen_total") == 0))

item_cols = ["recording_mbid", "bpm", "danceability", "onset_rate", "key_key", "key_scale",
             "tuning_frequency", "average_loudness", "dynamic_complexity", "mfcc_zero_mean"]

cold_items = (cold_pos.select(*item_cols)
              .dropDuplicates(["recording_mbid"])
              .withColumn("cidx", F.row_number().over(Window.orderBy("recording_mbid")) - 1))
n_cold_items = cold_items.count()

cand = (cold_pos.select("user_id", F.col("recording_mbid").alias("src_mbid"),
                        "listen_count_total", "distinct_recordings")
        .withColumn("k", F.explode(F.array(*[F.lit(i) for i in range(K_COLD)])))
        .withColumn("cidx", F.pmod(F.hash("user_id", "src_mbid", "k"), F.lit(n_cold_items)))
        .join(cold_items, "cidx")
        .drop("cidx", "k", "src_mbid")
        .dropDuplicates(["user_id", "recording_mbid"]))

# never call something "not played" if that listener actually played it
played_pairs = raw_df.filter("played = 1").select("user_id", "recording_mbid")
cold_neg = (cand.join(played_pairs, ["user_id", "recording_mbid"], "left_anti")
            .withColumn("played", F.lit(0))
            .withColumn("play_count", F.lit(0))
            .withColumn("log1p_play_count", F.lit(0.0))
            .withColumn("first_listen", F.lit(None))
            .withColumn("last_listen", F.lit(None))
            .withColumn("split", F.lit("test"))
            .withColumn("recording_listen_total", F.lit(0))
            .withColumn("distinct_listeners", F.lit(0))
            .withColumn("days_since_first_listen", F.lit(None)))
cold_neg = cold_neg.select([F.col(c).cast(raw_df.schema[c].dataType) for c in raw_df.columns])

df = raw_df.unionByName(cold_neg)

# every row gets one evaluation role
df = df.withColumn("eval_set",
        F.when(F.col("split") == "train", "train")
         .when(F.col("recording_listen_total") > 0, "test_warm")
         .otherwise("test_cold"))

print(f"brand-new recordings in the test period: {n_cold_items:,}")
df.groupBy("eval_set", "played").count().orderBy("eval_set", "played").show()

## 2. Clean the features and remove the popularity leak

- **Leave-one-out popularity.** For a training positive, the item's listen counts include that listener's own plays. Negatives never get that bonus, so the model would learn from it. Subtract the row's own contribution.
- **bpm of 0** is a failed beat tracker, not a tempo. Treat it as missing and fill with the training median.
- **Clip** danceability, onset_rate and dynamic_complexity to their 1st to 99th percentiles (danceability tops out at 14 on a scale that mostly sits near 1).
- **Log** the popularity counts, which are extremely skewed.

Every statistic here (percentiles, medians, means) comes from the training rows only.

In [ ]:
is_train_pos = (F.col("eval_set") == "train") & (F.col("played") == 1)

df = (df
  .withColumn("item_listens",   F.greatest(F.lit(0), F.when(is_train_pos, F.col("recording_listen_total") - F.col("play_count"))
                                                      .otherwise(F.col("recording_listen_total"))))
  .withColumn("item_listeners", F.greatest(F.lit(0), F.when(is_train_pos, F.col("distinct_listeners") - 1)
                                                      .otherwise(F.col("distinct_listeners"))))
  # if nobody else heard it before the split, it has no real "first listen" date either
  .withColumn("days_since_first_listen", F.when(F.col("item_listeners") == 0, None)
                                          .otherwise(F.col("days_since_first_listen").cast("double")))
  .withColumn("bpm", F.when(F.col("bpm") <= 0, None).otherwise(F.col("bpm")))
  .withColumn("key_key", F.when(F.col("key_key").isNull() | (F.trim(F.col("key_key")) == ""), "unknown")
                          .otherwise(F.col("key_key")))
  .withColumn("is_major", (F.col("key_scale") == "major").cast("double"))
  .withColumn("label", F.col("played").cast("double")))

train_only = df.filter("eval_set = 'train'")

# clip bounds and medians from TRAIN only
clip_cols = ["danceability", "onset_rate", "dynamic_complexity"]
bounds = dict(zip(clip_cols, train_only.approxQuantile(clip_cols, [0.01, 0.99], 0.001)))
med_bpm, med_days = [q[0] for q in train_only.approxQuantile(["bpm", "days_since_first_listen"], [0.5], 0.001)]

for c, (lo, hi) in bounds.items():
  df = df.withColumn(c, F.least(F.greatest(F.col(c), F.lit(lo)), F.lit(hi)))

df = (df
  .withColumn("bpm_missing", F.col("bpm").isNull().cast("double"))
  .withColumn("bpm", F.coalesce(F.col("bpm"), F.lit(med_bpm)))
  .withColumn("days_missing", F.col("days_since_first_listen").isNull().cast("double"))
  .withColumn("days_since_first_listen", F.coalesce(F.col("days_since_first_listen"), F.lit(med_days)))
  .withColumn("log_item_listens",    F.log1p("item_listens"))
  .withColumn("log_item_listeners",  F.log1p("item_listeners"))
  .withColumn("log_user_listens",    F.log1p("listen_count_total"))
  .withColumn("log_user_recordings", F.log1p("distinct_recordings"))
  # two folds for out-of-fold scoring, hash-based so they never change between runs
  .withColumn("fold", F.pmod(F.hash("user_id", "recording_mbid"), F.lit(2))))

train_only = df.filter("eval_set = 'train'")

# key as one-hot (fit on train)
key_indexer = StringIndexer(inputCol="key_key", outputCol="key_idx", handleInvalid="keep").fit(train_only)
key_ohe = OneHotEncoder(inputCols=["key_idx"], outputCols=["key_vec"], handleInvalid="keep").fit(key_indexer.transform(train_only))
df = key_ohe.transform(key_indexer.transform(df))

# standardized audio profile, used by the content engine (means and stds from TRAIN only)
cbf_src = ["bpm", "danceability", "onset_rate", "average_loudness", "dynamic_complexity", "mfcc_zero_mean", "is_major"]
st = train_only.agg(*[F.mean(c).alias("m_" + c) for c in cbf_src],
                    *[F.stddev(c).alias("s_" + c) for c in cbf_src]).first()
for c in cbf_src:
  df = df.withColumn("z_" + c, (F.col(c) - F.lit(st["m_" + c])) / F.lit(st["s_" + c] or 1.0))
cbf_cols = ["z_" + c for c in cbf_src]

df = df.cache()
print(f"rows after feature prep: {df.count():,}")
print("clip bounds:", {k: [round(v, 3) for v in b] for k, b in bounds.items()})
print(f"median bpm = {med_bpm:.1f}, median days_since_first_listen = {med_days:.0f}")
print("training positives whose recording had no OTHER listener before the split:",
      df.filter(is_train_pos & (F.col("item_listeners") == 0)).count())

## 3. Collaborative filtering engine (ALS) with out-of-fold scores

ALS learns a taste vector for every listener and recording from **training positives only** (implicit feedback, `log(1 + play_count)` as the strength). The dot product of the two vectors is `als_score`.

The catch: an ALS model that has seen a row will score it high, so using those scores as a training feature leaks the answer. So the training rows are split into two folds. Fold 0 rows are scored by an ALS model fit on fold 1, and the other way round. Test rows are scored by an ALS model fit on all training positives. No row is ever scored by a model that trained on it.

If ALS has never seen the listener or the recording, there is no score. `als_missing` = 1 marks those rows, which is the cold-start case CF can't handle.

In [ ]:
# ALS needs integer ids
user_indexer = StringIndexer(inputCol="user_id", outputCol="uidx", handleInvalid="keep").fit(df)
item_indexer = StringIndexer(inputCol="recording_mbid", outputCol="iidx", handleInvalid="keep").fit(df)
df_ids = (item_indexer.transform(user_indexer.transform(df))
          .withColumn("uidx", F.col("uidx").cast("int"))
          .withColumn("iidx", F.col("iidx").cast("int"))
          .withColumn("als_rating", F.log1p(F.col("play_count").cast("double"))))

als = ALS(userCol="uidx", itemCol="iidx", ratingCol="als_rating", implicitPrefs=True,
          rank=16, maxIter=10, regParam=0.1, alpha=10.0,
          coldStartStrategy="nan", seed=SEED)

train_pos = df_ids.filter((F.col("eval_set") == "train") & (F.col("played") == 1))
key = ["user_id", "recording_mbid"]

def score_with_als(fit_rows, score_rows):
  model = als.fit(fit_rows.select("uidx", "iidx", "als_rating"))
  return (model.transform(score_rows.select(*key, "uidx", "iidx"))
               .select(*key, F.col("prediction").alias("als_score")))

t0 = time.time()
als_scores = (score_with_als(train_pos.filter("fold = 1"), df_ids.filter("eval_set = 'train' AND fold = 0"))
  .unionByName(score_with_als(train_pos.filter("fold = 0"), df_ids.filter("eval_set = 'train' AND fold = 1")))
  .unionByName(score_with_als(train_pos, df_ids.filter("eval_set != 'train'"))))

prev = df
df = (df.join(als_scores, key, "left")
        .withColumn("als_missing", (F.col("als_score").isNull() | F.isnan("als_score")).cast("double"))
        .withColumn("als_score", F.when(F.col("als_missing") == 1, 0.0).otherwise(F.col("als_score"))))
df = df.cache(); df.count(); prev.unpersist()
print(f"ALS (3 fits) done in {time.time() - t0:.0f}s")

df.groupBy("eval_set", "played").agg(
    F.count("*").alias("rows"),
    F.round(1 - F.avg("als_missing"), 3).alias("share_with_als_score"),
    F.round(F.avg(F.when(F.col("als_missing") == 0, F.col("als_score"))), 4).alias("avg_als_score_when_present")
  ).orderBy("eval_set", "played").show()

## 4. Content-based engine (audio similarity)

A listener's taste profile is the average standardized audio profile (tempo, danceability, onset rate, loudness, dynamics, timbre, major/minor) of what they played. `cbf_score` is the cosine similarity between that profile and a recording's own audio profile.

This is the engine that still works on a brand-new recording, because it only needs the recording's audio. Same out-of-fold rule as ALS: a training row's taste profile is built from the other fold.

In [ ]:
def user_taste(pos_rows):
  return pos_rows.groupBy("user_id").agg(*[F.avg(c).alias("u_" + c) for c in cbf_cols])

def score_with_cbf(pos_rows, score_rows):
  r = score_rows.select(*key, *cbf_cols).join(user_taste(pos_rows), "user_id", "left")
  dot    = sum(F.col(c) * F.col("u_" + c) for c in cbf_cols)
  norm_u = F.sqrt(sum(F.col("u_" + c) * F.col("u_" + c) for c in cbf_cols))
  norm_i = F.sqrt(sum(F.col(c) * F.col(c) for c in cbf_cols))
  return r.select(*key, (dot / (norm_u * norm_i)).alias("cbf_score"))

pos_rows = df.filter((F.col("eval_set") == "train") & (F.col("played") == 1))
t0 = time.time()
cbf_scores = (score_with_cbf(pos_rows.filter("fold = 1"), df.filter("eval_set = 'train' AND fold = 0"))
  .unionByName(score_with_cbf(pos_rows.filter("fold = 0"), df.filter("eval_set = 'train' AND fold = 1")))
  .unionByName(score_with_cbf(pos_rows, df.filter("eval_set != 'train'"))))

prev = df
df = (df.join(cbf_scores, key, "left")
        .withColumn("cbf_missing", (F.col("cbf_score").isNull() | F.isnan("cbf_score")).cast("double"))
        .withColumn("cbf_score", F.when(F.col("cbf_missing") == 1, 0.0).otherwise(F.col("cbf_score"))))
df = df.cache(); df.count(); prev.unpersist()
print(f"CBF done in {time.time() - t0:.0f}s")

df.groupBy("eval_set", "played").agg(
    F.round(1 - F.avg("cbf_missing"), 3).alias("share_with_cbf_score"),
    F.round(F.avg(F.when(F.col("cbf_missing") == 0, F.col("cbf_score"))), 4).alias("avg_cbf_score_when_present")
  ).orderBy("eval_set", "played").show()

## 5. Score-level blend: sweeping alpha

`blend = alpha * CF + (1 - alpha) * CBF`. Both engines are turned into percentiles first (0 to 1), so a raw ALS score and a cosine similarity sit on the same footing.

Switching rule: when ALS has no score for a row, its CF part is 0, the bottom of the CF ranking. So among those rows only content decides the order, and they rank below rows where CF found real evidence.

Alpha is picked on the **training** rows, never on test, using ROC-AUC. ROC-AUC scores the whole ranking, including the rows CF knows nothing about. PR-AUC mostly watches the top of the list and doesn't notice that pure CF leaves most rows in one big tie at zero. alpha = 0 is pure content, alpha = 1 is pure CF.

In [ ]:
from pyspark.ml.feature import QuantileDiscretizer

train_only = df.filter("eval_set = 'train'")

def add_percentile(d, score, missing, out, when_missing):
  # percentile of the score, measured on TRAIN rows where the engine actually produced one
  has = train_only.filter(F.col(missing) == 0)
  if has.count() < 100:
    return d.withColumn(out, F.lit(when_missing))
  qd = QuantileDiscretizer(numBuckets=100, inputCol=score, outputCol=out + "_bucket", relativeError=0.001).fit(has)
  top = max(len(qd.getSplits()) - 2, 1)
  return (qd.transform(d)
            .withColumn(out, F.when(F.col(missing) == 1, F.lit(when_missing)).otherwise(F.col(out + "_bucket") / top))
            .drop(out + "_bucket"))

# no ALS score = no CF evidence, so it sits at the bottom of the CF ranking
# no taste profile = no content evidence either way, so it sits in the middle
df = add_percentile(df, "als_score", "als_missing", "cf_pct", 0.0)
df = add_percentile(df, "cbf_score", "cbf_missing", "cbf_pct", 0.5)

def blend_col(alpha):
  return F.lit(alpha) * F.col("cf_pct") + F.lit(1 - alpha) * F.col("cbf_pct")

roc_on_score = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="score", metricName="areaUnderROC")
sweep_rows = df.filter("eval_set = 'train'").select("label", "cf_pct", "cbf_pct").cache()

sweep = []
for a in [i / 10 for i in range(11)]:
  s = roc_on_score.evaluate(sweep_rows.withColumn("score", blend_col(a)))
  sweep.append((a, s))
  print(f"alpha = {a:.1f}   train ROC-AUC = {s:.4f}")
sweep_rows.unpersist()

best_alpha = max(sweep, key=lambda x: x[1])[0]
print(f"\nbest alpha on training data = {best_alpha}")
df = df.withColumn("blend_score", blend_col(best_alpha))

## 6. Train the models

All seven are trained on the same training rows with the same class weights. The classes are about 20/80, so each class gets half the total weight; that way a 0.5 threshold is a sensible cut for F1.

The four single-engine models are wrapped in a one-feature logistic regression. That doesn't change how they rank (so their AUC is the engine's own AUC), it just gives every model a probability and a threshold, so F1 is comparable across all seven.

In [ ]:
p_train = df.filter("eval_set = 'train'").agg(F.avg("label")).first()[0]
df = df.withColumn("w", F.when(F.col("label") == 1, 0.5 / p_train).otherwise(0.5 / (1 - p_train)))

audio_feats  = ["bpm", "bpm_missing", "danceability", "onset_rate", "tuning_frequency", "average_loudness",
                "dynamic_complexity", "mfcc_zero_mean", "is_major", "key_vec"]
pop_feats    = ["log_item_listens", "log_item_listeners", "days_since_first_listen", "days_missing"]
user_feats   = ["log_user_listens", "log_user_recordings"]
engine_feats = ["als_score", "als_missing", "cbf_score", "cbf_missing"]
all_feats = audio_feats + pop_feats + user_feats + engine_feats

def lr():
  return LogisticRegression(labelCol="label", weightCol="w", maxIter=50, regParam=0.001)

model_specs = [
  ("Popularity baseline",            ["log_item_listens"],          lr()),
  ("CF only (ALS)",                  ["als_score", "als_missing"],  lr()),
  ("CBF only (audio similarity)",    ["cbf_score", "cbf_missing"],  lr()),
  (f"Score blend (alpha={best_alpha})", ["blend_score"],             lr()),
  ("Hybrid: Logistic Regression",    all_feats,                     lr()),
  ("Hybrid: Random Forest",          all_feats,
      RandomForestClassifier(labelCol="label", weightCol="w", numTrees=60, maxDepth=10, seed=SEED)),
  ("Hybrid: Gradient-Boosted Trees", all_feats,
      GBTClassifier(labelCol="label", weightCol="w", maxIter=40, maxDepth=5, seed=SEED)),
]

train_df = df.filter("eval_set = 'train'").cache()
print(f"training rows: {train_df.count():,}   positive rate: {p_train:.3f}")

fitted = {}
for name, cols, algo in model_specs:
  t0 = time.time()
  fitted[name] = Pipeline(stages=[VectorAssembler(inputCols=cols, outputCol="features"), algo]).fit(train_df)
  print(f"trained {name:<34} {time.time() - t0:6.0f}s")

## 7. Evaluate

- **Warm test**: recordings that existed before the split. This is the main comparison: F1 (at 0.5), ROC-AUC and PR-AUC. PR-AUC is the one to trust most, because the classes are imbalanced on purpose.
- **Ranking**: Precision@10 and NDCG@10 per listener, only for listeners with at least 10 test candidates and one real play. On small data only a few hundred listeners qualify, so treat these as rough there. On big data almost every listener does.
- **Cold recordings**: brand-new recordings vs. the matched negatives from section 1. Popularity is zero for every row here.
- **New listeners**: known recordings, but a listener with no history before the split.

In [ ]:
auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="p1", metricName="areaUnderROC")
pr_eval  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="p1", metricName="areaUnderPR")

test_warm = df.filter("eval_set = 'test_warm'").cache()
test_cold = df.filter("eval_set = 'test_cold'").cache()
print(f"warm test rows: {test_warm.count():,}   cold test rows: {test_cold.count():,}")

def has_both_classes(d):
  return d.select("label").distinct().count() == 2

# an AUC needs both classes present; check each slice once, not once per model
slice_ok = {"warm": has_both_classes(test_warm),
            "cold": has_both_classes(test_cold),
            "new":  has_both_classes(test_warm.filter("listen_count_total = 0"))}

def safe(evaluator, pred, slice_name):
  return evaluator.evaluate(pred) if slice_ok[slice_name] else float("nan")

def f1_at_half(pred):
  c = pred.agg(
      F.sum(((F.col("prediction") == 1) & (F.col("label") == 1)).cast("int")).alias("tp"),
      F.sum(((F.col("prediction") == 1) & (F.col("label") == 0)).cast("int")).alias("fp"),
      F.sum(((F.col("prediction") == 0) & (F.col("label") == 1)).cast("int")).alias("fn")).first()
  tp, fp, fn = c["tp"] or 0, c["fp"] or 0, c["fn"] or 0
  precision = tp / (tp + fp) if tp + fp else 0.0
  recall = tp / (tp + fn) if tp + fn else 0.0
  f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
  return precision, recall, f1

def ranking_at_k(pred, k=10):
  ranked = (pred.withColumn("r", F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("p1"))))
                .withColumn("r_ideal", F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("label")))))
  per_user = ranked.groupBy("user_id").agg(
      F.count("*").alias("n"), F.sum("label").alias("npos"),
      F.sum(F.when(F.col("r") <= k, F.col("label")).otherwise(0.0)).alias("hits"),
      F.sum(F.when(F.col("r") <= k, F.col("label") / F.log2(F.col("r") + 1)).otherwise(0.0)).alias("dcg"),
      F.sum(F.when(F.col("r_ideal") <= k, F.col("label") / F.log2(F.col("r_ideal") + 1)).otherwise(0.0)).alias("idcg"))
  e = (per_user.filter((F.col("n") >= k) & (F.col("npos") >= 1))
               .agg(F.count("*").alias("users"), F.avg(F.col("hits") / k).alias("p"),
                    F.avg(F.col("dcg") / F.col("idcg")).alias("ndcg")).first())
  return e["users"], e["p"], e["ndcg"]

results = []
for name, model in fitted.items():
  t0 = time.time()
  pw = (model.transform(test_warm).withColumn("p1", vector_to_array("probability")[1])
             .select("user_id", "label", "prediction", "p1", "listen_count_total").cache())
  pc = (model.transform(test_cold).withColumn("p1", vector_to_array("probability")[1])
             .select("label", "p1").cache())
  precision, recall, f1 = f1_at_half(pw)
  users, p10, ndcg10 = ranking_at_k(pw)
  results.append({
      "Model": name,
      "F1": f1, "Precision": precision, "Recall": recall,
      "ROC-AUC": safe(auc_eval, pw, "warm"), "PR-AUC": safe(pr_eval, pw, "warm"),
      "P@10": p10 if users else float("nan"), "NDCG@10": ndcg10 if users else float("nan"),
      "Cold ROC-AUC": safe(auc_eval, pc, "cold"), "Cold PR-AUC": safe(pr_eval, pc, "cold"),
      "New-listener ROC-AUC": safe(auc_eval, pw.filter("listen_count_total = 0"), "new"),
  })
  pw.unpersist(); pc.unpersist()
  print(f"evaluated {name:<34} {time.time() - t0:6.0f}s")

ranking_users = users
res = pd.DataFrame(results).set_index("Model")
pd.set_option("display.width", 200)
print(f"\nlisteners eligible for P@10 / NDCG@10: {ranking_users or 0}")
res.round(4)

In [ ]:
# Copy-paste block for the project document (warm test set, threshold 0.5)
print("MODEL EVALUATION RESULTS (" + data_file_name + ")\n")
for m, r in res.iterrows():
  print(f"{m} -> F1: {r['F1']:.3f}, ROC-AUC: {r['ROC-AUC']:.3f}, PR-AUC: {r['PR-AUC']:.3f}")

print("\nCold-start test (brand-new recordings):")
for m, r in res.iterrows():
  print(f"{m} -> ROC-AUC: {r['Cold ROC-AUC']:.3f}, PR-AUC: {r['Cold PR-AUC']:.3f}")

best = res["PR-AUC"].idxmax()
print(f"\nBest model on warm PR-AUC: {best} ({res.loc[best, 'PR-AUC']:.3f})")
print(f"Best model on cold ROC-AUC: {res['Cold ROC-AUC'].idxmax()} ({res['Cold ROC-AUC'].max():.3f})")
print(f"Popularity baseline PR-AUC: {res.loc['Popularity baseline', 'PR-AUC']:.3f}")

## 8. What the best tree model leans on

In [ ]:
def importances(model_name, top=15):
  pipe = fitted[model_name]
  meta = pipe.stages[0].transform(train_df.limit(1)).schema["features"].metadata["ml_attr"]["attrs"]
  names = [a["name"] for a in sorted([a for grp in meta.values() for a in grp], key=lambda a: a["idx"])]
  imp = pipe.stages[-1].featureImportances.toArray()
  return pd.Series(imp, index=names).sort_values(ascending=False).head(top).round(4)

print("Gradient-Boosted Trees, top features:")
print(importances("Hybrid: Gradient-Boosted Trees").to_string())

## 9. Recommendations for one listener

Take one listener from the test period, rank their candidate recordings with the best model, and see where the ones they actually played land.

In [ ]:
best_model = fitted[best]
demo_user = (test_warm.groupBy("user_id").agg(F.sum("label").alias("plays"), F.count("*").alias("n"))
                      .filter("plays >= 1").orderBy(F.desc("n"), "user_id").first()["user_id"])

(best_model.transform(test_warm.filter(F.col("user_id") == demo_user))
   .withColumn("p_play", F.round(vector_to_array("probability")[1], 3))
   .select("recording_mbid", "p_play", "played", F.round("bpm", 0).alias("bpm"), "key_key", "key_scale",
           F.round("log_item_listens", 2).alias("log_popularity"))
   .orderBy(F.desc("p_play"))
   .show(10, truncate=False))
print(f"listener {demo_user}: played = 1 marks recordings they actually listened to after the split")